### Learning Instruction Finetuning

In [1]:
%load_ext autoreload
# %load_ext tensorboard
%autoreload 2

In [2]:
# ! pip install dotenv

In [3]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys

In [4]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning


In [5]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "rivertam" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [6]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [8]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [11]:
from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

    ## Fix this import
    
    # raise Exception("Not Running on Pace")
        # from src.model import GPT

In [12]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 8
0 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
1 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
2 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
3 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
4 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
5 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
6 NVIDIA GeForce RTX 2080 Ti 10.6 GiB
7 NVIDIA GeForce RTX 2080 Ti 10.6 GiB


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [13]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [14]:
# alpaca = AlpacaFormatOld(data)

In [15]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [16]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [17]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [18]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'role': 'user', 'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend'}], 'completion': [{'role': 'assistant', 'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [19]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [20]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [21]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [22]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [23]:

# inputs_1 = [0, 1, 2, 3, 4]
# inputs_2 = [5, 6]
# inputs_3 = [7, 8, 9]

# batch = (
#     inputs_1,
#     inputs_2,
#     inputs_3
# )
# custom_collate_fn(batch, pad_token_id=pad_token_id)

In [24]:
# logits_1 = torch.tensor(
#     [[-1.0, 1.0],  # 1st training example
#      [-0.5, 1.5]]  # 2nd training example
# )
# targets_1 = torch.tensor([0, 1])


# loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
# print(loss_1)

# logits_2 = torch.tensor(
#     [[-1.0, 1.0],
#      [-0.5, 1.5],
#      [-0.5, 1.5]]  # New 3rd training example
# )
# targets_2 = torch.tensor([0, 1, 1])

# loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
# print(loss_2)
# targets_3 = torch.tensor([0, 1, -100])

# loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
# print(loss_3)
# print("loss_1 == loss_3:", loss_1 == loss_3)

### Creating, Training and Testing Dataset

In [25]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [26]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)


train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [27]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [28]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [29]:
config =load_model_config(dataset_max_len=1024)
# MODEL_NAME = 'gpt2'

model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# import inspect
# inspect.signature(tokenizer.encode)

In [30]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens = in_tokens.to(next(model.parameters()).device)
print(in_tokens.shape)  # [1, sequence_length]

torch.Size([1, 4])


In [31]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

Every effort moves you one step forward toward your greatest potential. In this moment you have a choice. What will you do?


You may be thinking, "This is not just a book, this is a way of life


### SFT-Training

In [32]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

4.09375


In [38]:
from model_training.framework.train.core.train import train_model, TrainComponents
from model_training.framework.train.core.config import TrainConfig

from model_training.framework.train.loss.sft import sft_forward, sft_loss
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

ImportError: cannot import name 'TrainComponents' from 'model_training.framework.train.core.train' (/coc/scratch/vchopra/learning_env/Deep-Learning/model_training/framework/train/core/train.py)

In [ ]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [ ]:
train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [ ]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 253, with 3,085,697,024 parameters
num non-decayed parameter tensors: 181, with 241,664 parameters
using fused AdamW: False


In [ ]:
from torch.utils.data import DataLoader, Subset
from model_training.framework.data.post_train.dataloader import CyclingDataLoader


debug_examples = 80  # 10 batches × batch_size 8

base_loader = train_loader.loader

small_loader = DataLoader(
    Subset(base_loader.dataset, range(debug_examples)),
    batch_size=base_loader.batch_size,
    collate_fn=base_loader.collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=base_loader.num_workers,
)

train_loader_subset = CyclingDataLoader(
    small_loader,
    seq_len=train_loader.seq_len,
)

In [ ]:
train_comp = TrainComponents(optimizer=optimizer, 
                            dataloader=train_loader, 
                            scheduler= cosine_lr, 
                            forward_step=sft_forward,
                            loss_fn=sft_loss
                            )
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: False
    lr: 1e-05
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: False
    lr: 1e-05
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.framework.data.post_train.dataloader.CyclingDataLoader object at 0x331d8c6e0>, scheduler=<function cosine_lr at 0x3910ce2a0>, forward_step=<function sft_forward at 0x3910cdf80>, loss_fn=<function sft_loss at 0x3910cdee0>)

#### Step-7:- Sample Run 

In [ ]:
print("[STAGE] SFT forward and loss")
full_batch = train_comp.dataloader.next_batch()
small_batch = (full_batch[0][:1, :64], full_batch[1][:1, :64])

In [ ]:
logits, labels = train_comp.forward_step(model, small_batch, device)

In [ ]:
loss = train_comp.loss_fn(logits, labels)

In [ ]:
assert logits.shape[:2] == labels.shape
assert loss.ndim == 0 and loss.requires_grad and torch.isfinite(loss)
print(f"✓ logits={tuple(logits.shape)} loss={loss.item():.4f} requires_grad={loss.requires_grad}")

### Final Run

In [ ]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [ ]:
print(train_config.accumulation_steps)

2


In [ ]:
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger

run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

logger = ExperimentLogger(
    project="sft-experiments",
    log_dir=str(project_root / "execution_results" / "runs" / run_name))

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/vedaangchopra/.netrc.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: vedaangchopra (vedaangchopra_gatech) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already 

In [ ]:
%tensorboard --logdir "{project_root}/execution_results/runs"
model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Current Device: mps
Model set to Training mode:True
step 0 | loss: 3.780476 | norm :439.57 | dt: 53912.32 ms | tok/sec:303.90


In [ ]:
# # text_1 = "What is your name ? "
# text_1 = "Every effort moves you"
# in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
#                             #   allowed_special = {"<|endoftext|>"}
#                               )
# in_tokens=in_tokens.to(device)
# print(in_tokens.shape)  # [1, sequence_length]


# out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
# print(token_ids_to_text(out_tokens, tokenizer))